In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
import importlib
import matplotlib as mpl

# Set tick direction globally
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.top'] = True
mpl.rcParams['ytick.right'] = True

mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True

import os
# os.environ["PATH"] += ":/usr/bin"
# mpl.rcParams['text.usetex'] = True
# mpl.rcParams["text.latex.preamble"]+= r"\usepackage{amsmath}"

mpl.rcParams["xtick.labelsize"]=13
mpl.rcParams["ytick.labelsize"]=13
mpl.rcParams["axes.labelsize"]=15
mpl.rcParams["axes.titlesize"]=15
mpl.rcParams["legend.fontsize"]=13



In [ ]:
import bemsolver as bem
import importlib
import numpy as np
import matplotlib.pyplot as plt


# path="/home/sjoerd-buitjes/University/Master-Thesis/BEM/Boundary-Element-Method/spheroid.msh"

path="/home/sjoerd-buitjes/University/Master-Thesis/BEM/Boundary-Element-Method/datafiles/mesh/elongated-mesh-fine/elongated_spheroid_N=320.mat"
# path="/home/sjoerd-buitjes/University/Master-Thesis/BEM/Boundary-Element-Method/datafiles/mesh/sphere_refinement/sphere_mesh_h=2.000000e-01.mat"
# path ="/home/sjoerd-buitjes/University/Master-Thesis/BEM/Boundary-Element-Method/datafiles/mesh/pipette/pipette_paper.mat"

mesh= bem.Mesh(path)
print(mesh.center)
# print((np.shape(mesh.isosurface)[0]-1)/2)
# mesh.isosurface = mesh.isosurface[:int((np.shape(mesh.isosurface)[0]+1)/2),:]
# print(np.sum(mesh.normals,axis=0))
mesh.plot_mesh()




In [ ]:

Nx=200 + 1
Ny=200 + 1
xlim=3
ylim=3

x = np.linspace(-xlim, xlim, Nx)
y = np.linspace(-ylim, ylim, Ny)

xg, yg =np.meshgrid(x,y)
zg=np.zeros_like(xg)

xg = xg.ravel()
yg = yg.ravel()
zg = zg.ravel()
Ng = np.shape(xg)[0]

points = np.vstack((xg, yg, zg)).T


interaction = bem.FlowStokes(mesh,points)

In [ ]:
# Background flow
U = np.zeros(3)

U[0] = 0
U[1] = 0
U[2] = 0

# Background vorticity
W = np.zeros(3)

gamma_dot=1



W[0] = 0
W[1] = 0
W[2] = -gamma_dot/2

# Rate of strain tensor
E = gamma_dot/2*np.array([[0,1,0],
                          [1,0,0],
                          [0,0,0]])
sys = bem.ResistanceProblem(mesh)

psi, force, torque = sys.solve(U,W,E)
print("Condition number:", np.linalg.cond(sys.MATRIX))
print(f"\nTotal Force: {force} \nTotal Torque: {torque}")
# print(psi[-50:])


In [ ]:
U_field =interaction.calc_vector_field(psi,U,W,E)
print(np.shape(U_field))

fig = interaction.plot_vector_field(x, y, U_field,vmax=4,quiver_density=8, vector_scale=100)
fig.set_size_inches(10,8)
ax = fig.axes[0]
# ax.plot(mesh.isosurface[:,0],mesh.isosurface[:,1],color='r')
# ax.set_xlim(-5,30)
# ax.set_ylabel("hoi")
plt.show()




In [ ]:
from scipy.linalg import lu_factor, lu_solve
import bemsolver as bem
from scipy.integrate import solve_ivp
import numpy as np

gamma_dot=0.5

def find_flow(x):
    U = np.zeros(3)

    U[0] = 0 #gamma_dot * x[1]
    U[1] = 0
    U[2] = 0

    # Background vorticity
    W = np.zeros(3)  

    W[0] = 0
    W[1] = 0
    W[2] = -gamma_dot/2

    # Rate of strain tensor
    E = gamma_dot/2*np.array([[0,1,0],
                              [1,0,0],
                              [0,0,0]])
    return U, W, E




    

# path="/home/sjoerd-buitjes/University/Master-Thesis/BEM/Boundary-Element-Method/spheroid.msh"
path="/home/sjoerd-buitjes/University/Master-Thesis/BEM/Boundary-Element-Method/datafiles/mesh/elongated-mesh-fine/elongated_spheroid_N=320.mat"
# path = "/home/sjoerd-buitjes/University/Master-Thesis/BEM/Boundary-Element-Method/datafiles/mesh/spheroid/spheroid_0005.mat"
# path = "/home/sjoerd-buitjes/University/Master-Thesis/BEM/Boundary-Element-Method/datafiles/mesh/spheroid/fine-b=0.5/jeffrey_mesh_very_fine_b=0.5.mat"
# path = "/home/sjoerd-buitjes/University/Master-Thesis/BEM/Boundary-Element-Method/datafiles/mesh/spheroid-pole-density/spheroid_mesh_b=1.mat"
# path ="/home/sjoerd-buitjes/University/Master-Thesis/BEM/Boundary-Element-Method/datafiles/mesh/sphere_refinement/sphere_mesh_h=1.000000e-01.mat"
# path="/home/sjoerd-buitjes/University/Master-Thesis/BEM/Boundary-Element-Method/datafiles/mesh/sphere_refinement/sphere_mesh_h=2.500000e-01.mat"

mesh=bem.Mesh(path)


initial_orientation = np.array([1,0,0])
initial_position    = np.array([0,0,0])

sys=bem.MobilityProblem(mesh,flow_function=find_flow,
                        initial_position=initial_position,
                        initial_orientation=initial_orientation,
                        particle_velocity=1)
dt=0.01
T=100

solution = sys.RBM_over_time(T,dt) 





In [ ]:
from dataclasses import asdict


np.savez("solution.npz", **asdict(solution))
# asdict(solution)["X"]   
loaded = np.load("solution.npz")
# print(loaded["time"])

open_sol = bem.Solution()

open_sol.__dict__.update(loaded)
print(open_sol)


In [ ]:
%matplotlib widget


import matplotlib.pyplot as plt
X_coords = solution.X[:,:3]
x_s, y_s, z_s = solution.X[:,:3].T

print(x_s)
print(z_s)

fig = plt.figure(figsize=(6,5))
ax = fig.add_subplot(projection='3d')
ax.plot(x_s, y_s, z_s, lw=2)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
# ax.legend()

def set_axes_equal(ax):
        # xmin,xmax=swimmer.flow.xbounds
        # ymin,ymax=swimmer.flow.ybounds
        # zmin,zmax=swimmer.flow.zbounds

        xs = np.array([ax.get_xlim3d(), ax.get_ylim3d(), ax.get_zlim3d()])
        ranges = xs[:,1] - xs[:,0]
        centers = np.mean(xs, axis=1)
        radius = 0.5 * ranges.max()
        ax.set_xlim3d(centers[0]-radius, centers[0]+radius)
        ax.set_ylim3d(centers[1]-radius, centers[1]+radius)
        ax.set_zlim3d(centers[2]-radius, centers[2]+radius)

        # ax.set_xlim3d(xmin, xmax)
        # ax.set_ylim3d(ymin,ymax)
# ax.set_zlim3d(zmin,zmax)

set_axes_equal(ax)
plt.tight_layout()
plt.show()

In [ ]:

Nx=200 + 1
Ny=200 + 1
xlim=3
ylim=3

x = np.linspace(-xlim, xlim, Nx)
y = np.linspace(-ylim, ylim, Ny)

xg, yg =np.meshgrid(x,y)
zg=np.zeros_like(xg)

xg = xg.ravel()
yg = yg.ravel()
zg = zg.ravel()
Ng = np.shape(xg)[0]

points = np.vstack((xg, yg, zg)).T


interaction_2 = bem.FlowStokes(mesh,points)

In [ ]:
import matplotlib.pyplot as plt
# from scipy.spatial.transform import Rotation as R

def rotate_BCs(Q, U, W, E):
    """
    Rotate the boundary conditions to particle frame.
    """

    U_body = Q.T @ U
    W_body = Q.T @ W
    E_body = Q.T @ E @ Q

    return U_body, W_body, E_body

iter=0

U, W, E = find_flow(solution.X[iter,:3])
Q = solution.rotation_matrices[iter]


U_body, W_body, E_body = rotate_BCs(Q, U, W, E)

psi, u , omega = solution.psi[iter], solution.u[iter], solution.omega[iter]

U_field =interaction_2.calc_vector_field(psi,U_body,W_body,E_body)

fig = interaction_2.plot_vector_field(x, y, U_field,vmax=4,quiver_density=8, vector_scale=100)

fig.set_size_inches(10,8)
ax = fig.axes[0]
ax.plot(mesh.isosurface[:,0],-mesh.isosurface[:,1],color='r')
# ax.set_xlim(-5,30)
# ax.set_ylabel("hoi")
plt.show()





In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import imageio.v2 as imageio

# === USER INPUT ===
savepath = "/home/sjoerd-buitjes/University/Master-Thesis/Master-Thesis-Project/videos/Jeffrey-orbit/Jeffrey-orbit_class-test.mp4"
tmp_dir = "frames"
fps = 10
quiver_density = 8
vector_scale = 100
vmax_factor = 1.2
time_indices = range(0, len(solution.time), 50)  # every 10th frame

max_mag = 5
# max_mag = np.max(np.linalg.norm(U_field,axis=1))

# === PREPARE TEMP DIRECTORY ===
os.makedirs(tmp_dir, exist_ok=True)
frame_files = []

for i, iter in enumerate(time_indices):
    if i % 10==0:
        print(f"Rendering frame {i+1}/{len(time_indices)}")

    Q = solution.rotation_matrices[iter]

    U, W, E = find_flow(solution.X[iter])
    U_body, W_body, E_body = rotate_BCs(Q, U, W, E)
    psi, u, omega = solution.psi[iter], solution.u[iter], solution.omega[iter]
    U_field = interaction_2.calc_vector_field(psi, U_body, W_body, E_body)

    fig = interaction_2.plot_vector_field(
        x, y, U_field, max_mag,
        quiver_density=quiver_density,
        vector_scale=vector_scale
    )
    fig.set_size_inches(10, 8)
    ax = fig.axes[0]
    ax.set_title(f"t = {solution.time[iter]:.2f} s")
    ax.plot(mesh.isosurface[:,0],-mesh.isosurface[:,1],color='r')

    frame_path = os.path.join(tmp_dir, f"frame_{i:04d}.png")
    fig.savefig(frame_path, dpi=150)
    plt.close(fig)
    frame_files.append(frame_path)

# === CREATE GIF ===
print("Creating video...")
frames = [imageio.imread(f) for f in frame_files]
imageio.mimsave(savepath, frames, fps=fps)

print(f"Animation saved to {savepath}")

import shutil
shutil.rmtree(tmp_dir)
# (Optional) clean up temporary frames
# for f in frame_files:
#     os.remove(f)
# os.rmdir(tmp_dir)


In [ ]:
# mesh.centroids.mean(axis=0)
print(u)
print(omega)
print(sys.surface_matrix @ psi)
print(sys.torque_matrix @ psi)

In [ ]:
px, py, pz = solution.X[:,3:].T

# cond=np.linalg.cond(M)*1e-16

fig,ax=plt.subplots(2,1)
fig.set_size_inches(12,10)
# ax[0].set_title("Numerical Drift in $p_z$ for Line Completion Flow")
ax[0].plot(solution.time,px,color='b')
# ax[1].plot(t,py,label=f"$\\gamma={round(swimmer.major_minor_ratio(),3)}$")
ax[0].set_ylabel(r"$p_x$")
ax[0].set_xlabel("$t$")
# ax[0].set_xlim(0,1)
# ax[1].legend()    

# ax[1].set_title("Numerical Drift in $p_z$ for Point Completion Flow")

ax[1].plot(solution.time,py,color='r')
# ax[2].plot(t,np.sqrt(px**2+py**2+pz**2),label=f"$\\gamma={round(swimmer.shape_coefficient(),3)}$")
ax[1].set_ylabel(r"$p_y$")
ax[1].set_xlabel("$t$")
# print(T/dt * 1e-18)
# print("Condition number:", np.linalg.cond(M))



In [ ]:
x_arr, y_arr, z_arr =X[:,:3].T

fig,ax=plt.subplots(3,1)
fig.set_size_inches(20,10)
# ax[0].set_title("Numerical Drift in $p_z$ for Line Completion Flow")
ax[0].plot(time,x_arr,color='b')
# ax[1].plot(t,py,label=f"$\\gamma={round(swimmer.major_minor_ratio(),3)}$")
ax[0].set_ylabel(r"$x$")
ax[0].set_xlabel("$t$")
# ax[0].set_xlim(0,1)
# ax[1].legend()    

# ax[1].set_title("Numerical Drift in $p_z$ for Point Completion Flow")

ax[1].plot(time,y_arr,color='r')
# ax[2].plot(t,np.sqrt(px**2+py**2+pz**2),label=f"$\\gamma={round(swimmer.shape_coefficient(),3)}$")
ax[1].set_ylabel(r"$y$")
ax[1].set_xlabel("$t$")

ax[2].plot(time,z_arr,color='g')
# ax[2].plot(t,np.sqrt(px**2+py**2+pz**2),label=f"$\\gamma={round(swimmer.shape_coefficient(),3)}$")
ax[2].set_ylabel(r"$z$")
ax[2].set_xlabel("$t$")

In [ ]:
np.empty(16)